In [1]:
import torch
from torch.utils.data import DataLoader
import csv
import reasoning_gym
import sys
from pathlib import Path
from shared_utils.load import get_tokenizer, configs_from_yaml
from early_exit.util import get_model
from transformers import StoppingCriteriaList, StopStringCriteria

sys.path.append("../")
sys.path.append("../../")
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")


import csv
import torch
from reasoning_gym import create_dataset
# Assuming other necessary imports (StopStringCriteria, etc.) are present in your environment

model_config_path = "../config_qwen.yaml"
# Changed output filename to reflect new dataset
output_file = "rg_eval_qwen_knights_knaves.csv" 

def generate_rg_dataset(dataset_name: str, count: int = 30, seed: int = 42):
    """Generic helper to load any reasoning_gym dataset."""
    dataset = create_dataset(
        dataset_name,
        size=count,
        seed=seed,
    )
    return list(dataset)

def evaluate(model_name: str,
             dataset_name: str,
             count: int = 30,
             seed: int = 42):
    
    tokenizer = get_tokenizer(model_name)
    stop_criteria = StoppingCriteriaList([
        StopStringCriteria(stop_strings=["The answer is", "\n\n"], tokenizer=tokenizer)
    ])
    config = configs_from_yaml(model_config_path, tokenizer.eos_token_id)
    model = get_model(model_name, config["model"], device)

    # Open file context
    with open(output_file, mode="w", newline="", encoding="utf-8") as f:
        writer = csv.writer(f)
        writer.writerow(["question", "expected_answer", "generated_text", "model_answer", "is_correct"])

        data = generate_rg_dataset(dataset_name, count=count, seed=seed)

        for item in data:
            question = item["question"]
            expected_answer = str(item["answer"])

            prompt = f"Question: {question}\nAnswer:"
            inputs = tokenizer(prompt, return_tensors="pt").to(model.device)

            # Indentation fixed: Logic now inside the file context manager
            with torch.no_grad():
                outputs = model.generate(
                    **inputs,
                    max_new_tokens=500,          
                    do_sample=True,              
                    temperature=0.6,             
                    top_p=0.95,
                    top_k=20,
                    stopping_criteria=stop_criteria,
                    pad_token_id=tokenizer.eos_token_id,
                )
                generated_text = tokenizer.decode(outputs[0], skip_special_tokens=True)
                # Simple parsing logic - might need adjustment depending on how verbose the model is
                model_answer = generated_text.replace(prompt, "").strip().split("\n")[0]

                is_correct = expected_answer.lower() in model_answer.lower()
                
                print(
                    f"question: {question}\n"
                    f"expected_answer: {expected_answer}\n"
                    f"generated_text: {generated_text}\n"
                    f"model_answer: {model_answer}\n"
                    f"is_correct: {is_correct}\n"
                )
                
                # Write inside the loop while file is open
                writer.writerow([question, expected_answer, generated_text, model_answer, is_correct])

# Updated function call
evaluate(
    model_name="Qwen/Qwen2.5-Math-7B", # Adjusted based on typical usage, revert to Qwen3-4B if you have access
    dataset_name="knights_knaves",     # Changed to knights_knaves
    count=2000,
    seed=46,
)

/home/ubuntu/.local/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
/usr/lib/python3/dist-packages/scipy/__init__.py:146: UserWarning: A NumPy version >=1.17.3 and <1.25.0 is required for this version of SciPy (detected version 1.26.4
  warnings.warn(f"A NumPy version >={np_minversion} and <{np_maxversion}"


ImportError: cannot import name 'PreTrainedModel' from 'transformers' (/home/ubuntu/.local/lib/python3.10/site-packages/transformers/__init__.py)

In [2]:
!pip install "numpy<2.0"

Defaulting to user installation because normal site-packages is not writeable


In [ ]:
import torch
from torch.utils.data import DataLoader
import csv
import reasoning_gym
import sys
from pathlib import Path
from shared_utils.load import get_tokenizer, configs_from_yaml
from early_exit.util import get_model
from transformers import StoppingCriteriaList, StopStringCriteria

sys.path.append("../")
sys.path.append("../../")
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model_config_path = "../config_qwen.yaml"
output_file = "rg_eval_qwen_aiw.csv" 


def generate_rg_dataset(dataset_name: str, count: int = 30, seed: int = 42):
    """Generic helper to load any reasoning_gym dataset."""
    dataset = reasoning_gym.create_dataset(
        dataset_name,
        size=count,
        seed=seed,
    )
    return list(dataset)


def evaluate(model_name: str,
             dataset_name: str,
             count: int = 30,
             seed: int = 42):
    
    tokenizer = get_tokenizer(model_name)
    stop_criteria = StoppingCriteriaList([
    StopStringCriteria(stop_strings=["The answer is", "\n\n"], tokenizer=tokenizer)
])
    config = configs_from_yaml(model_config_path, tokenizer.eos_token_id)
    model = get_model(model_name, config["model"], device)

    with open(output_file, mode="w", newline="", encoding="utf-8") as f:
        writer = csv.writer(f)
        writer.writerow(["question", "expected_answer", "generated_text", "model_answer", "is_correct"])

        data = generate_rg_dataset(dataset_name, count=count, seed=seed)

        for item in data:
            question = item["question"]
            expected_answer = str(item["answer"])
        
            prompt = f"Question: {question}\nAnswer:"
            inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
        
            with torch.no_grad():
                outputs = model.generate(
                    **inputs,
                    max_new_tokens=500,
                    do_sample=True,
                    temperature=0.6,
                    top_p=0.95,
                    top_k=20,
                    stopping_criteria=stop_criteria,
                    pad_token_id=tokenizer.eos_token_id,
                )
                generated_text = tokenizer.decode(outputs[0], skip_special_tokens=True)
                model_answer = generated_text.replace(prompt, "").strip().split("\n")[0]
        
                is_correct = expected_answer.lower() in model_answer.lower()
                print(
                    f"question: {question}\n"
                    f"expected_answer: {expected_answer}\n"
                    f"generated_text: {generated_text}\n"
                    f"model_answer: {model_answer}\n"
                    f"is_correct: {is_correct}\n"
                )
                writer.writerow([question, expected_answer, generated_text, model_answer, is_correct])

evaluate(
    model_name="Qwen/Qwen3-4B",
    dataset_name="aiw",
    count=4000,
    seed=46,
)

In [68]:
import csv
import reasoning_gym

output_file = "rg_questions_answers_aiw.csv"

def generate_rg_dataset(dataset_name: str, count: int = 30, seed: int = 42):
    dataset = reasoning_gym.create_dataset(
        dataset_name,
        size=count,
        seed=seed,
    )  # each entry has "question" and "answer" fields [web:2][web:8]
    return list(dataset)

def save_questions_answers(dataset_name: str, count: int = 30, seed: int = 42):
    data = generate_rg_dataset(dataset_name, count=count, seed=seed)

    with open(output_file, mode="w", newline="", encoding="utf-8") as f:
        writer = csv.writer(f)
        writer.writerow(["question", "answer"])
        for item in data:
            question = item["question"]
            answer = str(item["answer"])
            writer.writerow([question, answer])

save_questions_answers(
    dataset_name="aiw",
    count=2000,
    seed=46,
)


In [1]:
import csv
import torch
from reasoning_gym import create_dataset
from transformers import (
    AutoTokenizer, 
    AutoModelForCausalLM, 
    StoppingCriteria, 
    StoppingCriteriaList
)

# --- Configuration ---
MODEL_NAME = "Qwen/Qwen2.5-Math-7B"
DATASET_NAME = "knights_knaves"
OUTPUT_FILE = "rg_eval_qwen_knights_knaves.csv"
COUNT = 2000
SEED = 46
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

# --- Helper Class for Stopping ---
class StopStringCriteria(StoppingCriteria):
    def __init__(self, tokenizer, stop_strings):
        self.tokenizer = tokenizer
        self.stop_strings = stop_strings

    def __call__(self, input_ids, scores, **kwargs):
        # Decode the last few tokens to check for stop string
        # This is a simplified check; for high-speed robust checking, token-id matching is better
        # but this works for basic scripts without external dependencies.
        text = self.tokenizer.decode(input_ids[0][-20:], skip_special_tokens=True)
        return any(s in text for s in self.stop_strings)

def generate_rg_dataset(dataset_name: str, count: int, seed: int):
    """Loads the dataset from reasoning_gym."""
    print(f"Generating {count} samples from {dataset_name}...")
    dataset = create_dataset(dataset_name, size=count, seed=seed)
    return list(dataset)

def evaluate():
    # 1. Load Model & Tokenizer (Standard Hugging Face)
    print(f"Loading model: {MODEL_NAME}...")
    tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, trust_remote_code=True)
    model = AutoModelForCausalLM.from_pretrained(
        MODEL_NAME, 
        device_map="auto", 
        torch_dtype=torch.bfloat16, # Use bfloat16 for Qwen to save memory
        trust_remote_code=True
    )

    # 2. Setup Stopping Criteria
    stop_criteria = StoppingCriteriaList([
        StopStringCriteria(tokenizer=tokenizer, stop_strings=["The answer is", "\n\n"])
    ])

    # 3. Generate Data
    data = generate_rg_dataset(DATASET_NAME, count=COUNT, seed=SEED)

    # 4. Run Evaluation Loop
    print(f"Starting evaluation. Output: {OUTPUT_FILE}")
    with open(OUTPUT_FILE, mode="w", newline="", encoding="utf-8") as f:
        writer = csv.writer(f)
        writer.writerow(["question", "expected_answer", "generated_text", "model_answer", "is_correct"])

        for i, item in enumerate(data):
            question = item["question"]
            expected_answer = str(item["answer"])

            prompt = f"Question: {question}\nAnswer:"
            inputs = tokenizer(prompt, return_tensors="pt").to(model.device)

            with torch.no_grad():
                outputs = model.generate(
                    **inputs,
                    max_new_tokens=500,
                    do_sample=True,
                    temperature=0.6,
                    top_p=0.95,
                    top_k=20,
                    stopping_criteria=stop_criteria,
                    pad_token_id=tokenizer.eos_token_id,
                )
                
            generated_text = tokenizer.decode(outputs[0], skip_special_tokens=True)
            
            # Basic parsing: Remove prompt, take first line of response
            response_only = generated_text[len(prompt):].strip()
            model_answer = response_only.split("\n")[0] if response_only else ""

            is_correct = expected_answer.lower() in model_answer.lower()

            # Print progress every 10 items
            if i % 10 == 0:
                print(f"[{i}/{COUNT}] Q: {question[:50]}... | Exp: {expected_answer} | Got: {model_answer} | {is_correct}")

            writer.writerow([question, expected_answer, generated_text, model_answer, is_correct])

if __name__ == "__main__":
    evaluate()

/home/ubuntu/.local/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
/usr/lib/python3/dist-packages/scipy/__init__.py:146: UserWarning: A NumPy version >=1.17.3 and <1.25.0 is required for this version of SciPy (detected version 1.26.4
  warnings.warn(f"A NumPy version >={np_minversion} and <{np_maxversion}"


Loading model: Qwen/Qwen2.5-Math-7B...


`torch_dtype` is deprecated! Use `dtype` instead!


ValueError: Could not find Qwen2ForCausalLM neither in <module 'transformers.models.qwen2' from '/home/ubuntu/.local/lib/python3.10/site-packages/transformers/models/qwen2/__init__.py'> nor in <module 'transformers' from '/home/ubuntu/.local/lib/python3.10/site-packages/transformers/__init__.py'>!

In [13]:
import csv
import reasoning_gym

# Configuration
output_file = "rg_leg_counting_dataset.csv"
dataset_name = "leg_counting"
count = 2000
seed = 46

def generate_and_save():
    print(f"Generating {count} samples from {dataset_name}...")
    
    # 1. Create Dataset
    dataset = reasoning_gym.create_dataset(
        dataset_name,
        size=count,
        seed=seed,
    )
    
    # 2. Write to CSV
    with open(output_file, mode="w", newline="", encoding="utf-8") as f:
        writer = csv.writer(f)
        # Header
        writer.writerow(["question", "expected_answer"])

        for item in dataset:
            question = item["question"]
            expected_answer = str(item["answer"])

            # Write row
            writer.writerow([question, expected_answer])

    print(f"Done. Saved {count} rows to {output_file}")

if __name__ == "__main__":
    generate_and_save()

Generating 2000 samples from leg_counting...
Done. Saved 2000 rows to rg_leg_counting_dataset.csv


AttributeError: module 'reasoning_gym' has no attribute 'list_datasets'